# ExplainDx - 01 Exploratory Data Analysis

Goal: understand the Kaggle *Disease Symptom Prediction* data **before** modelling, and prove why de-duplication is mandatory.

Run `python -m src.download_data` first (or place the CSVs in `data/raw/`).

In [ ]:
import sys, logging
from pathlib import Path

# Make `import src` work whether Jupyter was started in the repo root or in notebooks/.
ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src import config, eda, preprocess
from src.utils import setup_logging
setup_logging(logging.WARNING)
pd.set_option('display.max_columns', 30)

## 1. The raw files

In [ ]:
raw = preprocess.load_raw_tables(config.RAW_DIR)
print('dataset.csv shape:', raw.dataset.shape)
raw.dataset.head()

Notice the leading spaces and underscores in symptom names (e.g. `' dischromic _patches'`) and trailing spaces in disease names. Left alone, these create duplicate features and break joins between the four CSVs.

In [ ]:
symptom_cols = [c for c in raw.dataset.columns if c.lower().startswith('symptom')]
raw_names = pd.unique(raw.dataset[symptom_cols].to_numpy().ravel())
raw_names = [n for n in raw_names if pd.notna(n)]
changed = pd.DataFrame({'raw': raw_names, 'cleaned': [preprocess.clean_symptom(n) for n in raw_names]})
print(len(raw_names), 'distinct raw spellings ->', changed['cleaned'].nunique(), 'distinct cleaned symptoms')
changed[changed['raw'].str.strip() != changed['cleaned']].head(25)

## 2. Duplicates (the leakage trap)

In [ ]:
d = eda.load_for_eda(config.RAW_DIR)
r = d['report']
print(f"raw rows: {r.n_raw} | exact duplicates: {r.n_exact_duplicates} ({r.n_exact_duplicates / r.n_raw:.1%}) | unique rows: {r.n_unique}")
print('symptom vectors shared by several diseases:', r.n_conflicting_vectors)

If we split *before* removing these duplicates, the same patient pattern appears in train and test, so any model memorises its way to ~100% accuracy. That number would say nothing about generalisation.

In [ ]:
display(Image(str(eda.plot_class_balance(d['y_raw'], d['y_u'], config.FIGURES_DIR))))

## 3. Symptoms per record and most common symptoms

In [ ]:
display(Image(str(eda.plot_symptoms_per_row(d['X_u'], config.FIGURES_DIR))))
display(Image(str(eda.plot_top_symptoms(d['X_u'], d['symptoms'], config.FIGURES_DIR))))

## 4. Which symptoms define which disease?

In [ ]:
display(Image(str(eda.plot_disease_symptom_heatmap(d['X_u'], d['y_u'], d['symptoms'], config.FIGURES_DIR))))

## 5. Which diseases look alike? (predicts model confusion)

In [ ]:
jac = eda.disease_jaccard(d['X_u'], d['y_u'])
fig_path, csv_path = eda.plot_disease_overlap(jac, config.FIGURES_DIR)
display(Image(str(fig_path)))
pd.read_csv(csv_path)

## 6. Notes for your report

- How many unique rows does each disease really have? (see `reports/dataset_stats.md` after running `python -m src.preprocess`)
- Which disease pairs share most symptoms? Check later whether the model's confusion matrix confuses exactly those.
- The data is synthetic: patterns are idealised lists, not real patient records, so accuracy here is **not** clinical reliability.